In [1]:
import math
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

# Set deterministic seeds for reproducibility
SEED = 1337
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

PyTorch Version: 2.14.0+cu126
CUDA Available : True


#### Part I — Demystifying Autograd Mechanics

What this cell does: Demonstrates how PyTorch constructs dynamic Directed Acyclic Graphs (DAGs) under the hood using scalar tensors with requires_grad=True.

Key Inner Mechanics to Note:
- requires_grad=True: Tells PyTorch to track all operations involving this tensor in the computational graph.
- grad_fn: Points to the backward derivative function associated with the operation that created the tensor.
- retain_graph=False: By default, PyTorch frees intermediate buffers after calling .backward() to save memory.

In [2]:
# Initialize leaf nodes with gradient tracking
x1 = torch.tensor([2.0], dtype=torch.float64, requires_grad=True)
x2 = torch.tensor([0.0], dtype=torch.float64, requires_grad=True)
w1 = torch.tensor([-3.0], dtype=torch.float64, requires_grad=True)
w2 = torch.tensor([1.0], dtype=torch.float64, requires_grad=True)
b  = torch.tensor([6.8813735870195432], dtype=torch.float64, requires_grad=True)

# Forward pass (building the graph)
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

# Backward pass (computes dO/dx1, dO/dw1, etc. via reverse-mode AD)
o.backward()

print("--- Autograd Scalar Graph Evaluation ---")
print(f"Forward Output o : {o.item():.6f}")
print(f"Gradient do/dx1  : {x1.grad.item():.6f}")
print(f"Gradient do/dw1  : {w1.grad.item():.6f}")
print(f"Gradient do/db   : {b.grad.item():.6f}")

--- Autograd Scalar Graph Evaluation ---
Forward Output o : 0.707107
Gradient do/dx1  : -1.500000
Gradient do/dw1  : 1.000000
Gradient do/db   : 0.500000


#### The Danger of Accumulated Gradients (zero_grad)
What this cell does: Illustrates why zeroing gradients is necessary in PyTorch training loops.
Key Takeaway: PyTorch accumulates (adds) gradients into .grad on every .backward() call (p.grad += new_grad). This is intentional to support memory-efficient gradient accumulation across micro-batches, but requires explicitly calling optimizer.zero_grad() or tensor.grad = None before each training iteration to prevent training divergence.

In [3]:
x = torch.tensor([2.0], requires_grad=True)

# Iteration 1
y = x ** 2
y.backward()
print(f"Pass 1 | x.grad (should be 2*2=4): {x.grad.item()}")

# Iteration 2 WITHOUT zeroing gradient
y = x ** 2
y.backward()
print(f"Pass 2 | x.grad WITHOUT zero_grad (accumulated 4+4): {x.grad.item()}")

# Fixing with zero_grad / setting to None
x.grad = None # PyTorch best practice (more memory efficient than setting to 0.0)
y = x ** 2
y.backward()
print(f"Pass 3 | x.grad AFTER resetting to None: {x.grad.item()}")

Pass 1 | x.grad (should be 2*2=4): 4.0
Pass 2 | x.grad WITHOUT zero_grad (accumulated 4+4): 8.0
Pass 3 | x.grad AFTER resetting to None: 4.0


#### Part II — Data Preparation for Bigram Character Language Modeling
What this cell does: Loads text data and prepares bi-directional vocabulary maps (stoi, itos).Key Architecture Note: We use a single special character . (index 0) to mark both the start and end of a word sequence, expanding our total vocabulary size to $N = 27$ (26 letters + .).

In [4]:
# Synthetic sample dataset matching Karpathy's Makemore structure
raw_text = "emma\nolivia\neva\nsophia\nisabella\nava\nmia\namelia\nharper\nevelyn"
words = raw_text.splitlines()

# Build character mapping
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)

# Build training pairs (xs = input char index, ys = target next char index)
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print(f"Total training pairs (N) : {xs.nelement()}")
print(f"Sample Input Indices (xs): {xs[:5]}")
print(f"Sample Target Indices(ys): {ys[:5]}")

Total training pairs (N) : 61
Sample Input Indices (xs): tensor([0, 3, 7, 7, 1])
Sample Target Indices(ys): tensor([3, 7, 7, 1, 0])


#### Tensor Vectorization & One-Hot Encoding <br> 
What this cell does: Converts discrete integer indices into one-hot vectors using F.one_hot.Mathematical Insight: Multiplying a one-hot vector $x \in \mathbb{R}^{1 \times 27}$ by a weight matrix $W \in \mathbb{R}^{27 \times 27}$ is mathematically equivalent to indexing directly into row $i$ of $W$ ($x \cdot W = W[i, :]$). One-hot encoding allows us to treat lookup operations as a differentiable linear layer ($y = xW + b$).

In [5]:
# Convert input tensor to float32 one-hot representation
xenc = F.one_hot(xs, num_classes=vocab_size).float()

print(f"One-Hot Tensor Shape: {xenc.shape}")
print(f"First input one-hot vector (index {xs[0].item()}):")
print(xenc[0])

One-Hot Tensor Shape: torch.Size([61, 15])
First input one-hot vector (index 0):
tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


#### Linear Layer, Softmax & Negative Log-Likelihood (NLL) Loss
What this cell does: Implements the complete forward pass of a single-layer neural network from scratch using explicit PyTorch operations:Logits ($z = xW$): Computes raw unnormalized scores.Softmax ($P = \frac{e^z}{\sum e^z}$): Exponentiates logits to positive counts and normalizes across rows to obtain a valid probability distribution.Cross-Entropy / NLL Loss: Plucks the model's predicted probabilities for the correct target classes ys and computes average negative log-likelihood:$$\text{Loss} = -\frac{1}{N} \sum_{i=1}^N \log(P(y_i \mid x_i))$$

In [6]:
# Initialize weight matrix with manual seed
g = torch.Generator().manual_seed(SEED)
W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

# 1. Forward Pass: Compute Logits
logits = xenc @ W # Shape: (N, 27)

# 2. Forward Pass: Softmax Normalization
counts = logits.exp() # Equivalent to e^z
probs = counts / counts.sum(1, keepdim=True) # Normalize along row axis

# 3. Compute Negative Log-Likelihood (NLL) Loss for target indices `ys`
num_samples = xs.nelement()
loss = -probs[torch.arange(num_samples), ys].log().mean()

print(f"Initial Random Model Loss: {loss.item():.4f}")
print(f"Expected Expected Loss (-log(1/27)): {-math.log(1/27):.4f}")

Initial Random Model Loss: 2.9919
Expected Expected Loss (-log(1/27)): 3.2958


#### Full Training Loop with L2 Regularization (Weight Decay)
What this cell does: Executes a complete training optimization loop with Stochastic Gradient Descent (SGD) and L2 regularization. <br> 
Key Inner Mechanics to Note:
- L2 Regularization (0.01 * (W**2).mean()): Adds a penalty on large weights. Pushing weights toward zero forces logits toward zero, which Softmax converts into uniform probabilities ($1/27$). This is the exact neural network equivalent of Laplace Smoothing ($N + 1$) in counting models.

In [7]:
# Re-initialize weights
g = torch.Generator().manual_seed(SEED)
W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

num_samples = xs.nelement()
learning_rate = 50.0

print("--- Starting Neural Network Optimization ---")
for epoch in range(100):
    # 1. Forward Pass
    xenc = F.one_hot(xs, num_classes=vocab_size).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    
    # Loss = Negative Log-Likelihood + L2 Regularization Penalty
    loss = -probs[torch.arange(num_samples), ys].log().mean() + 0.01 * (W**2).mean()
    
    # 2. Zero Gradients
    W.grad = None
    
    # 3. Backward Pass
    loss.backward()
    
    # 4. SGD Parameter Update
    W.data += -learning_rate * W.grad
    
    if epoch % 20 == 0 or epoch == 99:
        print(f"Epoch {epoch:2d} | Loss: {loss.item():.4f}")

--- Starting Neural Network Optimization ---
Epoch  0 | Loss: 3.0028
Epoch 20 | Loss: 1.2372
Epoch 40 | Loss: 1.2298
Epoch 60 | Loss: 1.2281
Epoch 80 | Loss: 1.2274
Epoch 99 | Loss: 1.2781


#### Probabilistic Sampling from Trained Neural Network
What this cell does: Samples new character sequences from the trained bigram neural network using torch.multinomial.

In [8]:
g = torch.Generator().manual_seed(SEED)

print("--- Synthetic Name Generation ---")
for i in range(5):
    out = []
    ix = 0 # Start token '.'
    
    while True:
        # Forward pass for current character
        xenc = F.one_hot(torch.tensor([ix]), num_classes=vocab_size).float()
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)
        
        # Sample next character index based on probability vector `p`
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        
        if ix == 0: # End token '.' reached
            break
            
    print(''.join(out[:-1]))

--- Synthetic Name Generation ---
mmemia
olabelliammamivava
miamma
ellyn
elrphiabelivar


### super().__init__():

PyTorch internal tracking breaks: nn.Module uses its own __init__ to set up internal dictionaries that track layers, parameters, and gradients.

Errors on training/moving to GPU: PyTorch will throw errors like AttributeError: cannot assign module before Module.__init__() call or fail to register your weights when you call .to('cuda').

Rule of thumb: Whenever you create a custom class that inherits from another class (especially in PyTorch or libraries like Django/FastAPI/Pydantic), always call super().__init__() as the very first line inside your __init__ method.

In [ ]:
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, config):
        super().__init__()  # <-- Crucial step!
        self.layer = nn.Linear(10, 5)